# 02 — Export and correct NEON reflectance

Use this notebook when acquisition is complete and you want to work through correction explicitly. These are the same stages called by `go_forth_and_multiply`: HDF5 to raw ENVI, flightline-specific correction JSON, then topo/BRDF-corrected ENVI.

## 1. Configure the completed acquisition

Use the same output root, product code, and flightline stem as notebooks 00 and 01. Keep `topo_fit_mode="scene"` unless your experiment explicitly validates another supported mode.

In [ ]:
from pathlib import Path
from pprint import pprint

from spectralbridge.pipelines.pipeline import (
    stage_apply_brdf_topo_correction,
    stage_build_and_write_correction_json,
    stage_export_envi_from_h5,
)

RUN = False
base_folder = Path("outputs/neon_notebook")
product_code = "DP1.30006.001"
flight_stem = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
topo_fit_mode = "scene"
flight_dir = base_folder / flight_stem

## 2. Run or resume the correction stages

The functions validate canonical outputs and reuse completed work. The `outputs` dictionary makes the paths easy to inspect in the next cell, similar to the structured result printed by the drone orchestrator.

In [ ]:
outputs = {}
if RUN:
    raw_img, raw_hdr = stage_export_envi_from_h5(
        base_folder, product_code, flight_stem
    )
    correction_json = stage_build_and_write_correction_json(
        base_folder=base_folder,
        product_code=product_code,
        flight_stem=flight_stem,
        raw_img_path=raw_img,
        raw_hdr_path=raw_hdr,
    )
    corrected_img, corrected_hdr = stage_apply_brdf_topo_correction(
        base_folder=base_folder,
        product_code=product_code,
        flight_stem=flight_stem,
        raw_img_path=raw_img,
        raw_hdr_path=raw_hdr,
        correction_json_path=correction_json,
        topo_fit_mode=topo_fit_mode,
    )
    outputs = {
        "raw_img": raw_img,
        "raw_hdr": raw_hdr,
        "correction_json": correction_json,
        "corrected_img": corrected_img,
        "corrected_hdr": corrected_hdr,
    }
    pprint(outputs)
else:
    print("Dry run. Confirm the canonical HDF5 exists, then set RUN = True.")

## 3. Check the correction outputs

Both ENVI products require matching `.img` and `.hdr` files. The JSON is flightline evidence used by this correction, not a global coefficient file to copy between scenes.

In [ ]:
if outputs:
    for name, path in outputs.items():
        print(f"{name:>18}: exists={path.exists()} | {path}")
else:
    existing = sorted(flight_dir.glob("*")) if flight_dir.exists() else []
    print(f"Existing flightline files: {len(existing)}")
    for path in existing:
        if path.suffix in {'.img', '.hdr', '.json'}:
            print(f"  {path.name}")

## 4. Continue

Inspect spatial results with notebook 05 before harmonizing them in notebook 03. Do not rename the corrected pair: downstream stages use the canonical filename contract to decide whether work is valid and restartable.